# GDUT: stock YOLO11n Stage1

**Input:** the existing GDUT preparation output containing `prepared_gdut`.

Train the stock YOLO11n from `yolo11n.pt` on the base training split. This uses the CLEAN Stage1 recipe: 640, batch 16, seed 3407, AdamW, lr 0.0003, momentum 0.9, warmup bias lr 0, 100 maximum epochs and patience 50.

**Output:** `/kaggle/working/stage1_runs/gdut_S1_yolo11n/`, including best/last checkpoints, validation metrics and `run_manifest.json`. Save a version with outputs and mount it in the YOLO-DCRCF notebook. The YOLO11n scale matches the baseline graph.

Use a fresh Kaggle T4 x2 session with Internet enabled. Set `DEVICE="0"` consistently across the training notebooks to use one GPU.


## Runtime and modules


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

os.environ["PYTHONHASHSEED"] = "3407"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONFAULTHANDLER"] = "1"
os.environ["YOLO_CONFIG_DIR"] = "/kaggle/working/yolo_settings"
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
        "--disable-pip-version-check",
        "-q",
        "ultralytics==8.4.115",
        "albumentations==2.0.8",
    ]
)
CODE_DIR = Path("/kaggle/working/stage1_baseline_code")
CODE_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(CODE_DIR))
os.environ["PYTHONPATH"] = str(CODE_DIR) + os.pathsep + os.environ.get("PYTHONPATH", "")


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/baseline_model.py
"""Importable model construction and DDP trainers for the GDUT baselines."""

import json
import os

from torch import nn
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import LOGGER, LOCAL_RANK, RANK
from ultralytics.utils.loss import BboxLoss, v8DetectionLoss
from ultralytics.utils.torch_utils import torch_distributed_zero_first

from baseline_modules import ASPP, C2fWithDWR, DWR, ModulatedDCNv2, SBA, nwd_loss_pixels
from plain_logging import configure_plain_logging

configure_plain_logging()
REVISION = "gdut_paper_protocol_v2"
BASELINES = {"yolo11_dcrcf": "yolo11n", "yolov8_dwr_aspp_nwd": "yolov8s"}


class CIoUNWDBboxLoss(BboxLoss):
    def __init__(self, reg_max, constant=12.8, nwd_weight=0.5):
        super().__init__(reg_max)
        if not 0 <= nwd_weight <= 1 or constant <= 0:
            raise ValueError("Invalid NWD mixture or normalization constant.")
        self.constant = constant
        self.nwd_weight = nwd_weight

    def forward(
        self,
        pred_dist,
        pred_bboxes,
        anchor_points,
        target_bboxes,
        target_scores,
        target_scores_sum,
        fg_mask,
        imgsz,
        stride,
    ):
        ciou, dfl = super().forward(
            pred_dist,
            pred_bboxes,
            anchor_points,
            target_bboxes,
            target_scores,
            target_scores_sum,
            fg_mask,
            imgsz,
            stride,
        )
        # Convert feature-grid box coordinates to input-image pixels.
        pixel_scale = stride.unsqueeze(0).expand(pred_bboxes.shape[0], -1, 4)[fg_mask]
        pred_pixels = pred_bboxes[fg_mask].float() * pixel_scale
        target_pixels = target_bboxes[fg_mask].float() * pixel_scale
        quality = target_scores[fg_mask].sum(-1, keepdim=True)
        nwd = (
            nwd_loss_pixels(pred_pixels, target_pixels, self.constant) * quality
        ).sum()
        nwd = nwd / target_scores_sum
        return (1 - self.nwd_weight) * ciou + self.nwd_weight * nwd, dfl


class PaperBaselineModel(DetectionModel):
    def info(self, detailed=False, verbose=True, imgsz=640):
        """Return layer, parameter and gradient counts."""
        if not verbose:
            return
        layers = sum(not module._modules for module in self.modules())
        parameters = sum(parameter.numel() for parameter in self.parameters())
        gradients = sum(p.numel() for p in self.parameters() if p.requires_grad)
        LOGGER.info(
            f"{self.baseline_id}: {layers} layers, {parameters:,} parameters, "
            f"{gradients:,} gradients; FLOPs not profiled"
        )
        return layers, parameters, gradients, 0.0

    def init_criterion(self):
        criterion = v8DetectionLoss(self)
        if self.baseline_id == "yolov8_dwr_aspp_nwd":
            criterion.bbox_loss = CIoUNWDBboxLoss(
                self.model[-1].reg_max, self.nwd_constant, self.nwd_weight
            ).to(next(self.parameters()).device)
        return criterion


def graph_layer(module, index, inputs=-1):
    module.i, module.f = index, inputs
    module.type = type(module).__name__
    module.np = sum(parameter.numel() for parameter in module.parameters())
    return module


def module_inventory(model):
    classes = (DWR, ASPP, ModulatedDCNv2, SBA)
    return {
        cls.__name__: [
            name for name, module in model.named_modules() if isinstance(module, cls)
        ]
        for cls in classes
    }


def resize_neck_input(block, channels):
    """Adapt the input projection after replacing concat with an SBA output."""
    previous = block.cv1
    block.cv1 = type(previous)(channels, previous.conv.out_channels, 1, 1)
    return block


def apply_baseline(model, baseline_id, nwd_constant=12.8, nwd_weight=0.5):
    if hasattr(model, "baseline_id") or hasattr(model, "research_variant"):
        raise ValueError("The baseline graph has already been constructed.")
    if baseline_id not in BASELINES:
        raise ValueError(baseline_id)
    old = list(model.model)
    if baseline_id == "yolov8_dwr_aspp_nwd":
        if len(old) != 23 or old[9].__class__.__name__ != "SPPF":
            raise ValueError("Expected the standard YOLOv8s graph.")
        for index in (2, 4, 6):
            model.model[index] = graph_layer(
                C2fWithDWR(old[index]), index, old[index].f
            )
        model.model[9] = graph_layer(
            ASPP(old[9].cv1.conv.in_channels, old[9].cv2.conv.out_channels), 9
        )
    else:
        if len(old) != 24 or old[10].__class__.__name__ != "C2PSA":
            raise ValueError("Expected the standard YOLO11 P3/P4/P5 graph.")
        channels = {i: old[i].cv2.conv.out_channels for i in (4, 6, 8)}
        for index in (6, 8):
            old[index] = graph_layer(
                ModulatedDCNv2(old[index].cv1.conv.in_channels, channels[index]), index
            )
        p3, p4, p5 = channels[4], channels[6], channels[8]
        n3 = old[16].cv2.conv.out_channels
        n4 = old[13].cv2.conv.out_channels
        n5 = old[22].cv2.conv.out_channels
        layers = old[:11] + [
            graph_layer(SBA(p5, p4, n4), 11, [10, 6]),
            graph_layer(resize_neck_input(old[13], n4), 12),
            graph_layer(SBA(n4, p3, n3), 13, [12, 4]),
            graph_layer(resize_neck_input(old[16], n3), 14),
            graph_layer(SBA(n3, n4, n4), 15, [14, 12]),
            graph_layer(resize_neck_input(old[19], n4), 16),
            graph_layer(SBA(n4, p5, n5), 17, [16, 10]),
            graph_layer(resize_neck_input(old[22], n5), 18),
            graph_layer(old[23], 19, [14, 16, 18]),
        ]
        model.model = nn.Sequential(*layers)
        model.save = sorted(
            {
                j
                for layer in layers
                for j in ([layer.f] if isinstance(layer.f, int) else layer.f)
                if j != -1
            }
        )
    model.__class__ = PaperBaselineModel
    model.baseline_id = baseline_id
    model.baseline_revision = REVISION
    model.nwd_constant, model.nwd_weight = nwd_constant, nwd_weight
    model.reconstruction_status = "paper_based_reimplementation_not_author_code"
    assert_inventory(model)
    return model


def assert_inventory(model):
    expected = {
        "yolo11_dcrcf": {"DWR": 0, "ASPP": 0, "ModulatedDCNv2": 2, "SBA": 4},
        "yolov8_dwr_aspp_nwd": {"DWR": 3, "ASPP": 1, "ModulatedDCNv2": 0, "SBA": 0},
    }[model.baseline_id]
    actual = {key: len(value) for key, value in module_inventory(model).items()}
    if actual != expected:
        raise RuntimeError(f"Architecture mismatch: {actual} != {expected}")
    if list(model.stride.cpu().tolist()) != [8, 16, 32]:
        raise RuntimeError("Detection strides must remain 8/16/32.")


class PaperBaselineTrainer(DetectionTrainer):
    baseline_id = "yolo11_dcrcf"

    def final_eval(self):
        """Finalize checkpoint files; the launcher validates them after DDP exits."""
        from ultralytics.engine.trainer import strip_optimizer

        with torch_distributed_zero_first(LOCAL_RANK):
            if RANK in {-1, 0}:
                latest = strip_optimizer(self.last) if self.last.exists() else {}
                if not self.best.exists():
                    raise FileNotFoundError(
                        f"No best checkpoint was saved: {self.best}"
                    )
                strip_optimizer(
                    self.best, updates={"train_results": latest.get("train_results")}
                )
                marker = dict(
                    baseline_id=self.baseline_id,
                    revision=REVISION,
                    epochs_completed=self.epoch + 1,
                    best_epoch=getattr(self.stopper, "best_epoch", None),
                    final_validation="deferred_to_single_process_fp32",
                )
                (self.save_dir / "training_complete.json").write_text(
                    json.dumps(marker, indent=2), encoding="utf-8"
                )
                LOGGER.info("Checkpoints saved. Final validation will run outside DDP.")

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = self.set_model_names_for_load(
            DetectionModel(
                cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose
            )
        )
        if weights is not None:
            raise ValueError(
                "This protocol trains a new baseline from YAML without Stage1 weights."
            )
        options = json.loads(os.environ.get("GDUT_BASELINE_OPTIONS", "{}"))
        apply_baseline(model, self.baseline_id, **options)
        report = {
            "baseline_id": self.baseline_id,
            "revision": REVISION,
            "rank": os.environ.get("RANK", "-1"),
            "nc": self.data["nc"],
            "modules": module_inventory(model),
            "strides": model.stride.tolist(),
            "nwd_constant": model.nwd_constant,
            "nwd_weight": model.nwd_weight,
            "box_loss": "CIoU+NWD" if self.baseline_id.startswith("yolov8") else "CIoU",
            "parameters": sum(p.numel() for p in model.parameters()),
        }
        self.save_dir.mkdir(parents=True, exist_ok=True)
        (self.save_dir / f"architecture_rank_{report['rank']}.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )
        print(json.dumps(report, indent=2), flush=True)
        return model


class DCRCFTrainer(PaperBaselineTrainer):
    baseline_id = "yolo11_dcrcf"


class DWRASPPTrainer(PaperBaselineTrainer):
    baseline_id = "yolov8_dwr_aspp_nwd"


TRAINERS = {"yolo11_dcrcf": DCRCFTrainer, "yolov8_dwr_aspp_nwd": DWRASPPTrainer}


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/baseline_modules.py
"""DCNv2, selective boundary aggregation, DWR and ASPP modules."""

import torch
from torch import nn
from torch.nn import functional as F
from torchvision.ops import deform_conv2d


class ConvBNReLU(nn.Sequential):
    def __init__(self, c1, c2, kernel=1, dilation=1, groups=1):
        padding = dilation * (kernel - 1) // 2
        super().__init__(
            nn.Conv2d(
                c1,
                c2,
                kernel,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=False,
            ),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=False),
        )


class DWR(nn.Module):
    """Song et al., Figure 2: 2c -> 3c -> (2c,c,c) -> 2c, residual."""

    def __init__(self, channels):
        super().__init__()
        if channels % 2:
            raise ValueError("DWR requires an even channel count.")
        half = channels // 2
        self.region = ConvBNReLU(channels, 3 * half, kernel=3)
        self.branches = nn.ModuleList(
            [
                nn.Conv2d(half, channels, 3, padding=1, groups=half, bias=False),
                nn.Conv2d(
                    half, half, 3, padding=3, dilation=3, groups=half, bias=False
                ),
                nn.Conv2d(
                    half, half, 3, padding=5, dilation=5, groups=half, bias=False
                ),
            ]
        )
        self.norm = nn.BatchNorm2d(2 * channels)
        self.project = nn.Conv2d(2 * channels, channels, 1, bias=False)

    def forward(self, x):
        groups = self.region(x).chunk(3, dim=1)
        features = torch.cat([branch(t) for branch, t in zip(self.branches, groups)], 1)
        return x + self.project(self.norm(features))


class C2fWithDWR(nn.Module):
    def __init__(self, c2f):
        super().__init__()
        self.c2f = c2f
        self.dwr = DWR(c2f.cv2.conv.out_channels)

    def forward(self, x):
        return self.dwr(self.c2f(x))


class ASPP(nn.Module):
    """Song et al., Figure 3: rates 6/12/18 and image-level pooling."""

    def __init__(self, c1, c2, hidden=256):
        super().__init__()
        self.branches = nn.ModuleList(
            [
                ConvBNReLU(c1, hidden),
                ConvBNReLU(c1, hidden, kernel=3, dilation=6),
                ConvBNReLU(c1, hidden, kernel=3, dilation=12),
                ConvBNReLU(c1, hidden, kernel=3, dilation=18),
            ]
        )
        # Global pooling branch: 1x1 convolution followed by ReLU.
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c1, hidden, 1, bias=True),
            nn.ReLU(inplace=False),
        )
        self.project = ConvBNReLU(5 * hidden, c2)

    def forward(self, x):
        pooled = F.interpolate(
            self.pool(x), size=x.shape[-2:], mode="bilinear", align_corners=False
        )
        return self.project(
            torch.cat([branch(x) for branch in self.branches] + [pooled], 1)
        )


class ModulatedDCNv2(nn.Module):
    """A modulated 3x3 deformable convolution with learned offsets and masks."""

    def __init__(self, c1, c2):
        super().__init__()
        self.offset_mask = nn.Conv2d(c1, 27, 3, padding=1)
        self.weight = nn.Parameter(torch.empty(c2, c1, 3, 3))
        self.norm = nn.BatchNorm2d(c2)
        self.act = nn.SiLU()
        nn.init.kaiming_uniform_(self.weight, a=5**0.5)
        nn.init.zeros_(self.offset_mask.weight)
        nn.init.zeros_(self.offset_mask.bias)

    def forward(self, x):
        original_dtype = x.dtype
        # Run the compiled deformable operator in float32 under AMP and FP16 validation.
        with torch.autocast(device_type=x.device.type, enabled=False):
            features = x.float()
            fields = F.conv2d(
                features,
                self.offset_mask.weight.float(),
                self.offset_mask.bias.float(),
                padding=1,
            )
            offset, mask_logits = fields[:, :18], fields[:, 18:]
            result = deform_conv2d(
                features,
                offset,
                self.weight.float(),
                padding=(1, 1),
                mask=mask_logits.sigmoid(),
            )
        return self.act(self.norm(result.to(original_dtype)))


class RecalibrationUnit(nn.Module):
    """Recalibrate two feature tensors with 3x3 sigmoid gates."""

    def __init__(self, channels):
        super().__init__()
        self.gate_a = nn.Conv2d(channels, channels, 3, padding=1)
        self.gate_b = nn.Conv2d(channels, channels, 3, padding=1)

    def forward(self, a, b):
        ga, gb = self.gate_a(a).sigmoid(), self.gate_b(b).sigmoid()
        return a * ga + (1.0 - ga) * b * gb


class SBA(nn.Module):
    """Two RAUs and a 1x1 projection; output resolution follows the second input."""

    def __init__(self, c1, c2, out_channels):
        super().__init__()
        self.first = nn.Conv2d(c1, out_channels, 1)
        self.second = nn.Conv2d(c2, out_channels, 1)
        self.rau_ab = RecalibrationUnit(out_channels)
        self.rau_ba = RecalibrationUnit(out_channels)
        self.project = nn.Conv2d(2 * out_channels, out_channels, 1)

    def forward(self, inputs):
        first, second = inputs
        a, b = self.first(first), self.second(second)
        a = F.interpolate(a, size=b.shape[-2:], mode="bilinear", align_corners=False)
        return self.project(torch.cat([self.rau_ab(a, b), self.rau_ba(b, a)], 1))


def nwd_loss_pixels(predicted, target, constant):
    """Gaussian Wasserstein box loss; xyxy coordinates and C use input-image pixels."""
    if constant <= 0:
        raise ValueError("NWD normalization constant must be positive.")
    pred_center = (predicted[..., :2] + predicted[..., 2:]) / 2
    true_center = (target[..., :2] + target[..., 2:]) / 2
    pred_half_size = (predicted[..., 2:] - predicted[..., :2]) / 2
    true_half_size = (target[..., 2:] - target[..., :2]) / 2
    delta = torch.cat([pred_center - true_center, pred_half_size - true_half_size], -1)
    distance = torch.linalg.vector_norm(delta.float(), dim=-1, keepdim=True)
    return -torch.expm1(-distance / constant)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/baseline_preflight.py
"""Synthetic checks for the baseline graph, losses, and checkpoint portability."""

import copy
import io
from types import SimpleNamespace

import torch
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import DEFAULT_CFG_DICT

from baseline_model import (
    BASELINES,
    CIoUNWDBboxLoss,
    apply_baseline,
    assert_inventory,
    module_inventory,
)
from baseline_modules import ModulatedDCNv2, nwd_loss_pixels


def check_nwd():
    identical = torch.tensor([[0.0, 0.0, 10.0, 10.0]], requires_grad=True)
    loss = nwd_loss_pixels(identical, identical.detach(), 12.8)
    torch.testing.assert_close(loss, torch.zeros_like(loss))
    loss.sum().backward()
    assert torch.isfinite(identical.grad).all()
    predicted = torch.tensor([[1.0, 0.0, 11.0, 10.0]], requires_grad=True)
    distance = nwd_loss_pixels(predicted, identical.detach(), 12.8)
    expected = 1 - torch.exp(torch.tensor(-1 / 12.8))
    torch.testing.assert_close(distance.squeeze(), expected)
    distance.sum().backward()
    assert predicted.grad[:, [0, 2]].sum() > 0

    # The same physical box pair at different pyramid strides must have equal NWD.
    pred_pixels = torch.tensor([[[1.0, 0.0, 11.0, 10.0], [1.0, 0.0, 11.0, 10.0]]])
    true_pixels = torch.tensor([[[0.0, 0.0, 10.0, 10.0], [0.0, 0.0, 10.0, 10.0]]])
    stride = torch.tensor([[8.0], [32.0]])
    criterion = CIoUNWDBboxLoss(1, constant=12.8, nwd_weight=1.0)
    computed, _ = criterion(
        torch.zeros(1, 2, 4),
        pred_pixels / stride,
        torch.zeros(2, 2),
        true_pixels / stride,
        torch.ones(1, 2, 5) / 5,
        torch.tensor(2.0),
        torch.ones(1, 2, dtype=torch.bool),
        torch.tensor([640.0, 640.0]),
        stride,
    )
    torch.testing.assert_close(computed, expected)


def check_deformable_operator(device="cpu"):
    module = ModulatedDCNv2(4, 8).to(device)
    features = torch.rand(2, 4, 8, 8, device=device, requires_grad=True)
    module(features).square().mean().backward()
    assert module.offset_mask.weight.grad is not None
    assert torch.isfinite(module.offset_mask.weight.grad).all()
    assert module.offset_mask.weight.grad.abs().sum() > 0
    if device != "cpu":
        module.eval().half()
        with torch.no_grad():
            output = module(features.detach().half())
        assert output.dtype == torch.float16 and torch.isfinite(output).all()


def check_graph(baseline_id):
    torch.manual_seed(3407)
    model = DetectionModel(BASELINES[baseline_id] + ".yaml", nc=5, verbose=False)
    model.args = SimpleNamespace(**DEFAULT_CFG_DICT)
    apply_baseline(model, baseline_id)
    assert_inventory(model)
    model.train()
    image = torch.rand(2, 3, 64, 64)
    batch = dict(
        img=image,
        batch_idx=torch.tensor([0.0, 1.0]),
        cls=torch.tensor([[0.0], [4.0]]),
        bboxes=torch.tensor([[0.5, 0.5, 0.3, 0.3], [0.4, 0.6, 0.2, 0.3]]),
    )
    loss, _ = model(batch)
    if not torch.isfinite(loss).all():
        raise RuntimeError("Synthetic training loss is nonfinite.")
    loss.sum().backward()
    missing = [
        name
        for name, p in model.named_parameters()
        if p.requires_grad and p.grad is None
    ]
    if missing:
        raise RuntimeError(f"Modules disconnected from the training loss: {missing}")
    model.eval()
    with torch.no_grad():
        expected = model(image)[0]
    assert expected.shape == (2, 9, 84)
    stream = io.BytesIO()
    torch.save(model, stream)
    stream.seek(0)
    restored = torch.load(stream, weights_only=False).eval()
    with torch.no_grad():
        torch.testing.assert_close(restored(image)[0], expected)
    fused = copy.deepcopy(restored).fuse(verbose=False)
    with torch.no_grad():
        torch.testing.assert_close(fused(image)[0], expected, atol=2e-4, rtol=1e-4)
        single = fused(torch.rand(1, 3, 640, 640))[0]
    assert single.shape == (1, 9, 8400)
    return dict(
        modules=module_inventory(model),
        parameters=sum(p.numel() for p in model.parameters()),
    )


def run_preflight(baseline_id=None, operator_device="cpu"):
    original_threads = torch.get_num_threads()
    try:
        torch.set_num_threads(2)
        check_nwd()
        selected = [baseline_id] if baseline_id else list(BASELINES)
        if "yolo11_dcrcf" in selected:
            check_deformable_operator(operator_device)
        reports = {key: check_graph(key) for key in selected}
        print("Baseline preflight passed:", reports, flush=True)
        return reports
    finally:
        torch.set_num_threads(original_threads)


if __name__ == "__main__":
    run_preflight()


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/custom_ablation_modules.py
"""CoordConv, RepConv, routed attention, and Focal-EIoU modules."""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def autopad(k: int, p: int | None = None, d: int = 1) -> int:
    """Return padding that preserves spatial shape for odd kernels."""
    if p is None:
        p = d * (k - 1) // 2
    return p


class ConvBNAct(nn.Module):
    """Convolution followed by batch normalization and SiLU."""

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 1,
        s: int = 1,
        p: int | None = None,
        g: int = 1,
        d: int = 1,
        act: bool = True,
    ) -> None:
        super().__init__()
        self.conv = nn.Conv2d(
            c1, c2, k, s, autopad(k, p, d), groups=g, dilation=d, bias=False
        )
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.bn(self.conv(x)))


class CoordConv(nn.Module):
    """
    CoordConv layer: append normalized x/y coordinate channels before a conv.

    """

    def __init__(
        self, c1: int, c2: int, k: int = 3, s: int = 1, with_r: bool = False
    ) -> None:
        super().__init__()
        self.with_r = with_r
        extra = 3 if with_r else 2
        self.conv = ConvBNAct(c1 + extra, c2, k=k, s=s)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, _, h, w = x.shape
        yy = (
            torch.linspace(-1.0, 1.0, h, device=x.device, dtype=x.dtype)
            .view(1, 1, h, 1)
            .expand(b, 1, h, w)
        )
        xx = (
            torch.linspace(-1.0, 1.0, w, device=x.device, dtype=x.dtype)
            .view(1, 1, 1, w)
            .expand(b, 1, h, w)
        )
        coords = [xx, yy]
        if self.with_r:
            rr = torch.sqrt(torch.clamp(xx.square() + yy.square(), min=0.0))
            coords.append(rr)
        return self.conv(torch.cat([x, *coords], dim=1))


class RepConv(nn.Module):
    """
    RepConv block with 3x3, 1x1, and optional identity branches during training.

    Call `switch_to_deploy()` before ONNX/TensorRT export. The deploy path is a
    single 3x3 Conv2d plus activation.
    """

    def __init__(
        self,
        c1: int,
        c2: int,
        k: int = 3,
        s: int = 1,
        deploy: bool = False,
        act: bool = True,
    ) -> None:
        super().__init__()
        assert k == 3, "This RepConv implementation fuses to a 3x3 kernel."
        self.deploy = deploy
        self.in_channels = c1
        self.out_channels = c2
        self.stride = s
        self.act = nn.SiLU(inplace=True) if act else nn.Identity()

        if deploy:
            self.rbr_reparam = nn.Conv2d(c1, c2, 3, s, 1, bias=True)
        else:
            self.rbr_dense = nn.Sequential(
                nn.Conv2d(c1, c2, 3, s, 1, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_1x1 = nn.Sequential(
                nn.Conv2d(c1, c2, 1, s, 0, bias=False),
                nn.BatchNorm2d(c2),
            )
            self.rbr_identity = nn.BatchNorm2d(c1) if c1 == c2 and s == 1 else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.deploy:
            return self.act(self.rbr_reparam(x))
        out = self.rbr_dense(x) + self.rbr_1x1(x)
        if self.rbr_identity is not None:
            out = out + self.rbr_identity(x)
        return self.act(out)

    @staticmethod
    def _fuse_conv_bn(
        branch: nn.Sequential | nn.BatchNorm2d, channels: int
    ) -> tuple[torch.Tensor, torch.Tensor]:
        if isinstance(branch, nn.Sequential):
            conv = branch[0]
            bn = branch[1]
            kernel = conv.weight
        else:
            bn = branch
            input_dim = channels
            kernel = torch.zeros(
                (input_dim, input_dim, 3, 3),
                device=bn.weight.device,
                dtype=bn.weight.dtype,
            )
            for i in range(input_dim):
                kernel[i, i, 1, 1] = 1.0

        std = torch.sqrt(bn.running_var + bn.eps)
        t = (bn.weight / std).reshape(-1, 1, 1, 1)
        fused_kernel = kernel * t
        fused_bias = bn.bias - bn.running_mean * bn.weight / std
        return fused_kernel, fused_bias

    @staticmethod
    def _pad_1x1_to_3x3(kernel: torch.Tensor) -> torch.Tensor:
        if kernel.size(2) == 3:
            return kernel
        return F.pad(kernel, [1, 1, 1, 1])

    def get_equivalent_kernel_bias(self) -> tuple[torch.Tensor, torch.Tensor]:
        k3, b3 = self._fuse_conv_bn(self.rbr_dense, self.out_channels)
        k1, b1 = self._fuse_conv_bn(self.rbr_1x1, self.out_channels)
        if self.rbr_identity is None:
            kid = torch.zeros_like(k3)
            bid = torch.zeros_like(b3)
        else:
            kid, bid = self._fuse_conv_bn(self.rbr_identity, self.out_channels)
        return k3 + self._pad_1x1_to_3x3(k1) + kid.to(k3.device), b3 + b1 + bid.to(
            b3.device
        )

    def switch_to_deploy(self) -> None:
        if self.deploy:
            return
        kernel, bias = self.get_equivalent_kernel_bias()
        self.rbr_reparam = nn.Conv2d(
            self.in_channels,
            self.out_channels,
            3,
            self.stride,
            1,
            bias=True,
        )
        self.rbr_reparam.weight.data = kernel.detach().clone()
        self.rbr_reparam.bias.data = bias.detach().clone()
        del self.rbr_dense
        del self.rbr_1x1
        if hasattr(self, "rbr_identity"):
            del self.rbr_identity
        self.deploy = True


class BiFormerBlockLite(nn.Module):
    """
    Region routing and fine token attention with a residual connection.

    Region-level routing selects the key/value tokens used by fine attention.
    The residual block uses four heads and eight-pixel regions by default.
    """

    def __init__(
        self, channels: int, num_heads: int = 4, region_size: int = 8, topk: int = 4
    ) -> None:
        super().__init__()
        assert channels % num_heads == 0, "channels must be divisible by num_heads"
        self.channels = channels
        self.num_heads = num_heads
        self.region_size = region_size
        self.topk = topk
        self.qkv = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.proj = nn.Conv2d(channels, channels, 1, bias=False)
        self.norm = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.shape
        rs = self.region_size
        pad_h = (rs - h % rs) % rs
        pad_w = (rs - w % rs) % rs
        x_pad = F.pad(x, (0, pad_w, 0, pad_h))
        hp, wp = x_pad.shape[-2:]
        gh, gw = hp // rs, wp // rs

        q, k, v = self.qkv(x_pad).chunk(3, dim=1)
        q_regions = q.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        k_regions = k.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()
        v_regions = v.unfold(2, rs, rs).unfold(3, rs, rs).contiguous()

        # [B, C, GH, GW, RS, RS] -> [B, R, T, C]
        q_tokens = q_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        k_tokens = k_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)
        v_tokens = v_regions.permute(0, 2, 3, 4, 5, 1).reshape(b, gh * gw, rs * rs, c)

        q_region = q_tokens.mean(dim=2)
        k_region = k_tokens.mean(dim=2)
        route_logits = torch.matmul(q_region, k_region.transpose(-1, -2)) / (c**0.5)
        topk = min(self.topk, gh * gw)
        route_idx = route_logits.topk(topk, dim=-1).indices

        out_regions = []
        head_dim = c // self.num_heads
        for region_idx in range(gh * gw):
            selected = route_idx[:, region_idx]
            k_sel = torch.stack(
                [
                    k_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            v_sel = torch.stack(
                [
                    v_tokens[bi, selected[bi]].reshape(topk * rs * rs, c)
                    for bi in range(b)
                ],
                dim=0,
            )
            q_cur = q_tokens[:, region_idx]

            qh = q_cur.reshape(b, rs * rs, self.num_heads, head_dim).transpose(1, 2)
            kh = k_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            vh = v_sel.reshape(b, topk * rs * rs, self.num_heads, head_dim).transpose(
                1, 2
            )
            attn = torch.softmax(
                torch.matmul(qh, kh.transpose(-1, -2)) / (head_dim**0.5), dim=-1
            )
            out = torch.matmul(attn, vh).transpose(1, 2).reshape(b, rs * rs, c)
            out_regions.append(out)

        y = torch.stack(out_regions, dim=1).reshape(b, gh, gw, rs, rs, c)
        y = y.permute(0, 5, 1, 3, 2, 4).reshape(b, c, hp, wp)
        y = y[:, :, :h, :w]
        return x + self.norm(self.proj(y))


def xywh_to_xyxy(boxes: torch.Tensor) -> torch.Tensor:
    x, y, w, h = boxes.unbind(-1)
    half_w = w / 2
    half_h = h / 2
    return torch.stack((x - half_w, y - half_h, x + half_w, y + half_h), dim=-1)


def focal_eiou_loss(
    pred_boxes: torch.Tensor,
    target_boxes: torch.Tensor,
    xywh: bool = True,
    gamma: float = 0.5,
    eps: float = 1e-7,
    reduction: str = "mean",
) -> torch.Tensor:
    """
    Focal-EIoU box regression loss.

    Args:
        pred_boxes: [N, 4] predicted boxes.
        target_boxes: [N, 4] target boxes.
        xywh: True when boxes are center-x, center-y, width, height.
        gamma: focal weighting exponent.
    """
    if xywh:
        pred = xywh_to_xyxy(pred_boxes)
        target = xywh_to_xyxy(target_boxes)
    else:
        pred = pred_boxes
        target = target_boxes

    px1, py1, px2, py2 = pred.unbind(-1)
    tx1, ty1, tx2, ty2 = target.unbind(-1)
    pw = (px2 - px1).clamp(min=eps)
    ph = (py2 - py1).clamp(min=eps)
    tw = (tx2 - tx1).clamp(min=eps)
    th = (ty2 - ty1).clamp(min=eps)

    inter_x1 = torch.maximum(px1, tx1)
    inter_y1 = torch.maximum(py1, ty1)
    inter_x2 = torch.minimum(px2, tx2)
    inter_y2 = torch.minimum(py2, ty2)
    inter = (inter_x2 - inter_x1).clamp(min=0) * (inter_y2 - inter_y1).clamp(min=0)
    union = pw * ph + tw * th - inter + eps
    iou = (inter / union).clamp(min=eps, max=1.0)

    pcx = (px1 + px2) / 2
    pcy = (py1 + py2) / 2
    tcx = (tx1 + tx2) / 2
    tcy = (ty1 + ty2) / 2
    center_dist = (pcx - tcx).square() + (pcy - tcy).square()

    cw = (torch.maximum(px2, tx2) - torch.minimum(px1, tx1)).clamp(min=eps)
    ch = (torch.maximum(py2, ty2) - torch.minimum(py1, ty1)).clamp(min=eps)
    c2 = cw.square() + ch.square() + eps

    eiou = (
        1.0
        - iou
        + center_dist / c2
        + (pw - tw).square() / (cw.square() + eps)
        + (ph - th).square() / (ch.square() + eps)
    )
    loss = iou.pow(gamma) * eiou

    if reduction == "mean":
        return loss.mean()
    if reduction == "sum":
        return loss.sum()
    if reduction == "none":
        return loss
    raise ValueError(f"Unsupported reduction: {reduction}")


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/gdut_data.py
"""Load the existing GDUT split without generating or augmenting samples."""

import hashlib
import json
from pathlib import Path

import yaml

NAMES = ["blue", "white", "yellow", "red", "none"]


def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def prepared_dataset(explicit=""):
    candidates = (
        [Path(explicit)]
        if explicit
        else [p.parent for p in Path("/kaggle/input").rglob("dataset_manifest.json")]
    )
    found = []
    for root in candidates:
        path = root / "dataset_manifest.json"
        if not path.is_file():
            continue
        meta = json.loads(path.read_text(encoding="utf-8"))
        if meta.get("dataset") == "gdut":
            found.append((root, meta, sha256(path)))
    if len(found) != 1:
        raise ValueError("Mount one GDUT preparation output or set PREPARED_DIR.")
    root, meta, fingerprint = found[0]
    if meta["names"] != NAMES:
        raise ValueError(
            "Expected the GDUT class order: blue, white, yellow, red, none."
        )
    if (
        meta.get("preparation_version", 1) >= 2
        and not (root / "preparation_complete.json").exists()
    ):
        raise ValueError("The preparation output is incomplete.")
    for split in ("train", "val", "test"):
        for kind in ("images", "labels"):
            if not (root / "base" / kind / split).is_dir():
                raise FileNotFoundError(root / "base" / kind / split)
    return root, meta, fingerprint


def data_yaml(root, output_directory):
    output_directory = Path(output_directory)
    output_directory.mkdir(parents=True, exist_ok=True)
    config = dict(
        path=str(root),
        names=dict(enumerate(NAMES)),
        train="base/images/train",
        val="base/images/val",
        test="base/images/test",
    )
    target = output_directory / "gdut.yaml"
    target.write_text(yaml.safe_dump(config), encoding="utf-8")
    return str(target)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/plain_logging.py
"""Plain-text formatting for Ultralytics console messages."""

import logging
import re
import sys
from functools import wraps


ANSI_ESCAPE = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")
DECORATIONS = re.compile(r"[\U0001f000-\U0001faff\u2600-\u27bf\ufe0f\u200d]")
REPLACEMENTS = str.maketrans(
    {
        "\u2192": "->",
        "\u2190": "<-",
        "\u2194": "<->",
        "\u2265": ">=",
        "\u2264": "<=",
        "\u00d7": "x",
        "\u2705": "[OK]",
        "\u2713": "[OK]",
        "\u274c": "[FAIL]",
        "\u2717": "[FAIL]",
    }
)


def plain_text(message):
    """Remove console decoration while preserving labels and numeric values."""
    message = ANSI_ESCAPE.sub("", message).translate(REPLACEMENTS)
    return DECORATIONS.sub("", message)


class PlainFormatter(logging.Formatter):
    def format(self, record):
        message = plain_text(super().format(record))
        if record.levelno >= logging.WARNING and not message.startswith(
            record.levelname + ":"
        ):
            message = re.sub(r"^" + record.levelname + r"\s+", "", message)
            message = f"{record.levelname}: {message}"
        return message


class ProgressStream:
    """Write progress updates as complete, undecorated notebook lines."""

    def __init__(self, stream):
        self.stream = stream

    def write(self, message):
        cleaned = plain_text(message)
        cleaned = re.sub(r"[\u2500-\u259f]", "-", cleaned)
        if "\r" in cleaned:
            cleaned = cleaned.replace("\r", "").rstrip()
            if cleaned:
                cleaned += "\n"
        self.stream.write(cleaned)
        return len(message)

    def flush(self):
        self.stream.flush()

    def __getattr__(self, name):
        return getattr(self.stream, name)


def configure_progress_output():
    """Format the pinned Ultralytics progress writer without changing iteration."""
    from ultralytics.utils import TQDM

    if getattr(TQDM.__init__, "_plain_progress", False):
        return
    original = TQDM.__init__

    @wraps(original)
    def initialize(self, *args, **kwargs):
        if len(args) > 4:
            args = list(args)
            args[4] = ProgressStream(args[4] or sys.stdout)
        else:
            kwargs["file"] = ProgressStream(kwargs.get("file") or sys.stdout)
        original(self, *args, **kwargs)
        self.mininterval = max(self.mininterval, 10.0)

    initialize._plain_progress = True
    TQDM.__init__ = initialize


def configure_plain_logging():
    """Keep the library's log levels and handlers, using plain-text output."""
    from ultralytics.utils import LOGGER

    for handler in LOGGER.handlers:
        handler.setFormatter(PlainFormatter("%(message)s"))
    configure_progress_output()


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/pretrained_conversion.py
"""Function-preserving Conv conversions; module topology is unchanged."""

import torch
from custom_ablation_modules import CoordConv, RepConv, BiFormerBlockLite


def copy_bn(source, target):
    target.load_state_dict(source.state_dict())
    # BatchNorm state_dict excludes eps and momentum.
    target.eps = source.eps
    target.momentum = source.momentum


@torch.no_grad()
def coord_from_conv(source):
    c = source.conv
    if c.groups != 1 or c.bias is not None:
        raise ValueError("Expected unfused standard Conv + BN")
    new = CoordConv(
        c.in_channels, c.out_channels, k=c.kernel_size[0], s=c.stride[0], with_r=False
    )
    new.to(device=c.weight.device, dtype=c.weight.dtype)
    new.conv.conv.weight.zero_()
    new.conv.conv.weight[:, : c.in_channels].copy_(c.weight)
    copy_bn(source.bn, new.conv.bn)
    return new


@torch.no_grad()
def rep_from_conv(source):
    c = source.conv
    if c.groups != 1 or c.bias is not None or c.kernel_size != (3, 3):
        raise ValueError("Expected unfused standard 3x3 Conv + BN")
    new = RepConv(c.in_channels, c.out_channels, k=3, s=c.stride[0], deploy=False)
    new.to(device=c.weight.device, dtype=c.weight.dtype)
    new.rbr_dense[0].weight.copy_(c.weight)
    copy_bn(source.bn, new.rbr_dense[1])
    # Keep new branches trainable; zero BN scale makes their initial contribution zero.
    new.rbr_1x1[1].weight.zero_()
    new.rbr_1x1[1].bias.zero_()
    if new.rbr_identity is not None:
        new.rbr_identity.weight.zero_()
        new.rbr_identity.bias.zero_()
    return new


@torch.no_grad()
def identity_start_attention(channels, source):
    new = BiFormerBlockLite(channels=channels, num_heads=4)
    p = next(source.parameters())
    new.to(device=p.device, dtype=p.dtype)
    new.norm.weight.zero_()
    new.norm.bias.zero_()
    # Zero residual scale starts the attention block as an identity map.
    return new


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/research_trainer.py
"""Model construction and distributed trainers for the ablation experiments."""

import json
import os

from plain_logging import configure_plain_logging

configure_plain_logging()

from torch import nn
from ultralytics.models.yolo.detect import DetectionTrainer
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils.loss import BboxLoss, v8DetectionLoss

from custom_ablation_modules import (
    BiFormerBlockLite,
    CoordConv,
    RepConv,
    focal_eiou_loss,
)
from pretrained_conversion import (
    coord_from_conv,
    identity_start_attention,
    rep_from_conv,
)

ARCHITECTURE_REVISION = "biformer_both_backbones_v2"
INITIALIZATION_POLICY = "preserve_pretrained_v1"


class SPPFWithBiFormer(nn.Module):
    """Retain the YOLOv8 SPPF and apply attention to its P5 output."""

    def __init__(self, sppf, attention):
        super().__init__()
        self.sppf = sppf
        self.attention = attention

    def forward(self, x):
        return self.attention(self.sppf(x))


class FocalEIoUBboxLoss(BboxLoss):
    """Use Focal-EIoU for box regression and retain the native DFL term."""

    def forward(
        self,
        pred_dist,
        pred_bboxes,
        anchor_points,
        target_bboxes,
        target_scores,
        target_scores_sum,
        fg_mask,
        imgsz,
        stride,
    ):
        _, loss_dfl = super().forward(
            pred_dist,
            pred_bboxes,
            anchor_points,
            target_bboxes,
            target_scores,
            target_scores_sum,
            fg_mask,
            imgsz,
            stride,
        )
        weight = target_scores[fg_mask].sum(-1)
        loss_box = focal_eiou_loss(
            pred_bboxes[fg_mask],
            target_bboxes[fg_mask],
            xywh=False,
            gamma=0.5,
            reduction="none",
        )
        return (loss_box * weight).sum() / target_scores_sum, loss_dfl


class ResearchDetectionModel(DetectionModel):
    def init_criterion(self):
        criterion = v8DetectionLoss(self)
        if self.research_variant in ("A4", "A6"):
            criterion.bbox_loss = FocalEIoUBboxLoss(self.model[-1].reg_max).to(
                next(self.parameters()).device
            )
        return criterion


def inventory(model):
    """Return top-level graph indices containing each custom module."""
    module_types = {
        "CoordConv": CoordConv,
        "RepConv": RepConv,
        "BiFormerBlockLite": BiFormerBlockLite,
    }
    return {
        name: [
            index
            for index, layer in enumerate(model.model)
            if any(isinstance(child, module_type) for child in layer.modules())
        ]
        for name, module_type in module_types.items()
    }


def replace(model, index, replacement):
    previous = model.model[index]
    replacement.i = getattr(previous, "i", index)
    replacement.f = getattr(previous, "f", -1)
    replacement.type = replacement.__class__.__name__
    replacement.np = sum(parameter.numel() for parameter in replacement.parameters())
    model.model[index] = replacement


def patch(model, variant, initialization=INITIALIZATION_POLICY):
    """Construct one ablation from a stock Stage1 model."""
    if variant not in ("A1", "A2", "A3", "A4", "A5", "A6"):
        raise ValueError(f"Unknown variant: {variant}")
    if initialization not in (INITIALIZATION_POLICY, "legacy_random"):
        raise ValueError(f"Unknown initialization policy: {initialization}")
    preserve = initialization == INITIALIZATION_POLICY
    classes = {layer.__class__.__name__ for layer in model.model}
    if "C2PSA" in classes:
        family = "yolo11"
    elif "C2f" in classes:
        family = "yolov8"
    else:
        raise ValueError("The ablations support YOLO11s and YOLOv8s only.")

    conversions = []
    if variant in ("A2", "A6"):
        previous = model.model[0]
        conv = previous.conv
        replacement = (
            coord_from_conv(previous)
            if preserve
            else CoordConv(conv.in_channels, conv.out_channels, k=3, s=2)
        )
        replace(model, 0, replacement)
        conversions.append(
            {"index": 0, "module": "CoordConv", "copied_conv_bn": preserve}
        )

    if variant in ("A3", "A6"):
        for index, layer in enumerate(model.model):
            if variant == "A6" and index == 0:
                continue
            conv = getattr(layer, "conv", None)
            if getattr(conv, "kernel_size", None) == (3, 3):
                replacement = (
                    rep_from_conv(layer)
                    if preserve
                    else RepConv(
                        conv.in_channels,
                        conv.out_channels,
                        k=3,
                        s=conv.stride[0],
                    )
                )
                replace(model, index, replacement)
                conversions.append(
                    {"index": index, "module": "RepConv", "copied_conv_bn": preserve}
                )

    if variant in ("A5", "A6"):
        index = 10 if family == "yolo11" else 9
        previous = model.model[index]
        expected_type = "C2PSA" if family == "yolo11" else "SPPF"
        if previous.__class__.__name__ != expected_type:
            raise ValueError(f"Expected {expected_type} at layer {index}.")
        channels = (
            previous.cv1.conv.in_channels
            if family == "yolo11"
            else previous.cv2.conv.out_channels
        )
        attention = (
            identity_start_attention(channels, previous)
            if preserve
            else BiFormerBlockLite(channels, num_heads=4)
        )
        replacement = (
            attention if family == "yolo11" else SPPFWithBiFormer(previous, attention)
        )
        replace(model, index, replacement)
        conversions.append(
            {
                "index": index,
                "module": "BiFormerBlockLite",
                "placement": "replace_C2PSA" if family == "yolo11" else "after_SPPF",
                "identity_start": preserve,
            }
        )

    rep_indices = (
        [0, 1, 3, 5, 7, 17, 20] if family == "yolo11" else [0, 1, 3, 5, 7, 16, 19]
    )
    expected = {
        "CoordConv": [0] if variant in ("A2", "A6") else [],
        "RepConv": rep_indices if variant in ("A3", "A6") else [],
        "BiFormerBlockLite": (
            [10 if family == "yolo11" else 9] if variant in ("A5", "A6") else []
        ),
    }
    if variant == "A6":
        expected["RepConv"] = expected["RepConv"][1:]
    if inventory(model) != expected:
        raise RuntimeError(f"Unexpected module inventory: {inventory(model)}")

    model.research_variant = variant
    model.research_family = family
    model.research_status = "as_defined"
    model.architecture_revision = ARCHITECTURE_REVISION
    model.initialization_policy = initialization
    model.initialization_audit = conversions
    return model


class ResearchTrainer(DetectionTrainer):
    variant = "A1"

    def get_model(self, cfg=None, weights=None, verbose=True):
        # Each DDP worker constructs the graph before loading custom weights.
        model = self.set_model_names_for_load(
            ResearchDetectionModel(
                cfg,
                nc=self.data["nc"],
                ch=self.data["channels"],
                verbose=verbose,
            )
        )
        if weights is not None and hasattr(weights, "research_variant"):
            if weights.research_variant != self.variant:
                raise ValueError("Each ablation must start from its Stage1 checkpoint.")
            if (
                weights.research_family == "yolov8"
                and self.variant in ("A5", "A6")
                and getattr(weights, "architecture_revision", None)
                != ARCHITECTURE_REVISION
            ):
                raise ValueError(
                    "This YOLOv8 checkpoint predates the BiFormer extension."
                )
            patch(
                model,
                self.variant,
                initialization=getattr(
                    weights, "initialization_policy", "legacy_random"
                ),
            )
            model.load(weights)
        else:
            if weights is not None:
                model.load(weights)
            patch(model, self.variant)

        report = {
            "variant": self.variant,
            "family": model.research_family,
            "status": model.research_status,
            "rank": os.environ.get("RANK", "-1"),
            "modules": inventory(model),
            "nc": self.data["nc"],
            "box_loss": "Focal-EIoU" if self.variant in ("A4", "A6") else "CIoU",
            "cls_loss": "BCE",
            "dfl": "native Ultralytics DFL",
            "parameters": sum(parameter.numel() for parameter in model.parameters()),
            "architecture_revision": ARCHITECTURE_REVISION,
            "initialization_policy": model.initialization_policy,
            "initialization_audit": model.initialization_audit,
        }
        self.save_dir.mkdir(parents=True, exist_ok=True)
        path = self.save_dir / f"architecture_rank_{report['rank']}.json"
        path.write_text(json.dumps(report, indent=2), encoding="utf-8")
        print(json.dumps(report, indent=2), flush=True)
        return model


class A1Trainer(ResearchTrainer):
    variant = "A1"


class A2Trainer(ResearchTrainer):
    variant = "A2"


class A3Trainer(ResearchTrainer):
    variant = "A3"


class A4Trainer(ResearchTrainer):
    variant = "A4"


class A5Trainer(ResearchTrainer):
    variant = "A5"


class A6Trainer(ResearchTrainer):
    variant = "A6"


TRAINERS = {
    trainer.variant: trainer
    for trainer in (A1Trainer, A2Trainer, A3Trainer, A4Trainer, A5Trainer, A6Trainer)
}


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/sitecustomize.py
"""Apply the notebook console format in distributed training workers."""

import os

if "LOCAL_RANK" in os.environ:
    from plain_logging import configure_plain_logging

    configure_plain_logging()


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_comparison.py
"""Compare Stage1 baseline runs and existing CLEAN A6 checkpoints on common splits."""

import argparse
import csv
import json
from pathlib import Path

from ultralytics import YOLO
import yaml

from baseline_model import assert_inventory
from gdut_data import prepared_dataset, sha256
from research_trainer import ARCHITECTURE_REVISION, inventory
from warm_data import dataset_yaml
from warm_metrics import benchmark_forward, export_metrics
from warm_protocol import COMMON, EVALUATION, MODELS, REVISION
from warm_runner import CODE_DIR, check_version, release, write_json


def csv_file(path, rows):
    with Path(path).open("w", newline="", encoding="utf-8-sig") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def inputs_for_comparison(input_root, dataset_hash, software_check=False):
    found = {}
    for path in Path(input_root).rglob("run_manifest.json"):
        value = json.loads(path.read_text())
        if value.get("dataset") != "gdut":
            continue
        if (
            value.get("warm_revision") == REVISION
            and value.get("baseline_id") in MODELS
        ):
            identifier = value["baseline_id"]
        elif value.get("stage") == "A6" and value.get("model") in (
            "yolo11s",
            "yolov8s",
        ):
            identifier = "A6_" + value["model"]
            if value.get("architecture_revision") != ARCHITECTURE_REVISION:
                raise ValueError(
                    "Use the original CLEAN A6 revision with BiFormer on both backbones."
                )
        else:
            continue
        if identifier in found:
            raise ValueError(f"Mount one output for {identifier}.")
        if value.get("smoke") and not software_check:
            raise ValueError("Use full experiment checkpoints.")
        if (
            value.get("dataset_fingerprint") != dataset_hash
            or value.get("imgsz") != 640
        ):
            raise ValueError("Dataset version or input size differs.")
        checkpoint = (path.parent / value["checkpoint"]).resolve()
        if not checkpoint.is_relative_to(path.parent.resolve()):
            raise ValueError("Checkpoints must remain inside their run directory.")
        if not checkpoint.is_file() or sha256(checkpoint) != value["checkpoint_sha256"]:
            raise ValueError("Missing or modified comparison checkpoint.")
        found[identifier] = (path, value, checkpoint)
    required = set(MODELS) | {"A6_yolo11s", "A6_yolov8s"}
    if found.keys() != required:
        raise ValueError(f"Missing inputs: {sorted(required-found.keys())}")
    baseline = found["yolov8_dwr_aspp_nwd"][1]
    reference = found["A6_yolov8s"][1]
    if baseline["source_checkpoint_sha256"] != reference["source_checkpoint_sha256"]:
        raise ValueError(
            "YOLOv8 baseline and A6 must start from the same stock Stage1 checkpoint."
        )
    left = found["yolo11_dcrcf"][1]["runtime"]
    right = baseline["runtime"]
    if left["hardware"] != right["hardware"] or left["software"] != right["software"]:
        raise ValueError("The two baseline training environments differ.")
    return found


def audit_recipe(found, software_check=False):
    rows = []
    for key, expected in COMMON.items():
        row = dict(parameter=key, configured_common=expected)
        known = []
        for identifier, (path, manifest, _) in found.items():
            saved = path.parent / "args.yaml"
            args = (
                yaml.safe_load(saved.read_text())
                if saved.is_file()
                else manifest.get("training_args", {})
            )
            value = args.get(key)
            row[identifier] = value
            if value is not None:
                known.append(value)
                if value != expected and not software_check:
                    raise ValueError(
                        f"Stage2 configuration differs: {identifier}/{key}."
                    )
        row["status"] = (
            "matched" if len(known) == len(found) else "not_recorded_in_all_runs"
        )
        rows.append(row)
    return rows


def compare(
    prepared_dir="",
    device="0",
    input_root="/kaggle/input",
    working="/kaggle/working",
    software_check=False,
):
    check_version()
    prepared, _, dataset_hash = prepared_dataset(prepared_dir)
    found = inputs_for_comparison(input_root, dataset_hash, software_check)
    recipe = audit_recipe(found, software_check)
    output = Path(working) / "stage1_baseline_comparison"
    if output.exists():
        raise FileExistsError(f"Use a fresh comparison directory: {output}")
    output.mkdir(parents=True)
    data = dataset_yaml(prepared, output / "configuration", False)
    rows = []
    for identifier, (path, metadata, weight) in sorted(found.items()):
        model = YOLO(str(weight))
        if identifier in MODELS:
            assert_inventory(model.model)
            if (
                model.model.warm_revision != REVISION
                or model.model.baseline_id != identifier
            ):
                raise ValueError("Baseline graph differs from its manifest.")
        elif inventory(model.model) != metadata["modules"]:
            raise ValueError("A6 graph differs from its manifest.")
        parameters = sum(p.numel() for p in model.model.parameters())
        results_file = path.parent / "results.csv"
        actual_epochs = metadata.get("epochs_completed")
        if actual_epochs is None and results_file.is_file():
            with results_file.open() as stream:
                actual_epochs = sum(1 for _ in csv.DictReader(stream))
        details = dict(
            manifest=metadata,
            source_manifest=str(path),
            checkpoint_sha256=sha256(weight),
            splits={},
        )
        current_rows = []
        for split in ("val", "test"):
            settings = dict(EVALUATION)
            if software_check:
                settings.update(imgsz=64, batch=2, plots=False)
            result = model.val(
                data=data,
                split=split,
                device=device,
                project=str(output),
                name=identifier + "_" + split,
                **settings,
            )
            metrics, classes, definitions = export_metrics(result)
            details["splits"][split] = dict(
                metrics=metrics, per_class=classes, definitions=definitions
            )
            current_rows.append(
                dict(
                    model=identifier,
                    backbone=metadata["model"],
                    Stage2_epochs_completed=actual_epochs,
                    Stage1_epochs_completed=metadata.get("source_manifest", {}).get(
                        "observed_epochs_completed"
                    ),
                    split=split,
                    parameters=parameters,
                    **metrics,
                )
            )
        timing = benchmark_forward(model.model, device, smoke=software_check)
        for row in current_rows:
            row.update(
                forward_ms_median=timing["forward_ms_median"],
                deployment_parameters=timing["deployment_parameters"],
            )
        rows.extend(current_rows)
        details["timing"] = timing
        write_json(output / (identifier + "_details.json"), details)
        del model
        release()
    references = json.loads((CODE_DIR / "paper_references.json").read_text())
    paper_rows = []
    for row in rows:
        if row["model"] not in references:
            continue
        paper = references[row["model"]]
        record = dict(
            model=row["model"],
            split=row["split"],
            paper_dataset=paper["dataset"],
            experiment_dataset="GDUT",
            doi=paper["doi"],
            comparison_type="cross_dataset",
        )
        for metric in ("precision", "recall", "mAP50", "mAP50_95"):
            record["paper_" + metric] = paper["metrics"][metric]
            record["experiment_" + metric] = row[metric]
            record[metric + "_delta_pp"] = 100 * (
                row[metric] - paper["metrics"][metric]
            )
        paper_rows.append(record)
    for split in ("val", "test"):
        csv_file(
            output / ("comparison_" + split + ".csv"),
            [row for row in rows if row["split"] == split],
        )
    csv_file(output / "paper_vs_experiment.csv", paper_rows)
    csv_file(output / "stage2_recipe_audit.csv", recipe)
    report = dict(
        complete=True,
        software_check=software_check,
        results=rows,
        paper_comparison=paper_rows,
        stage2_recipe=COMMON,
        dataset_fingerprint=dataset_hash,
        yolo8_same_stage1_checkpoint=True,
        yolo11_scale_comparison="YOLO11n baseline versus YOLO11s A6; separate scale-matched stock Stage1 sources",
        hardware={
            key: value[1].get("runtime", "not recorded in original manifest")
            for key, value in found.items()
        },
        selection="best validation mAP50-95; original early stopping policy",
        training_budget="Configured epochs and early stopping match; realized Stage1/Stage2 epochs can differ",
    )
    write_json(output / "comparison_report.json", report)
    lines = [
        "# Baseline theo protocol Stage1 của A6 CLEAN",
        "",
        "## 2.1. Thực nghiệm",
        "",
        "Hai baseline dùng Stage1 GDUT theo scale tương ứng, sau đó AdamW, lr 0.0001, ảnh 640, batch 16, seed 3407, tối đa 100 epoch, patience 50 và base + hardaug. Kiến trúc và loss baseline giữ nguyên. YOLOv8 dùng cùng hash Stage1 với A6; YOLO11n dùng Stage1 riêng vì A6 là YOLO11s. Số epoch thực tế và ngân sách Stage1 được ghi riêng, không coi là bằng nhau khi early stopping khác nhau.",
        "",
        "## 2.2. Paper và thực nghiệm",
        "",
        "paper_vs_experiment.csv chứa số công bố SHWD và số GDUT theo từng split, đơn vị 0–1 và delta theo điểm phần trăm. comparison_val.csv và comparison_test.csv tách kết quả của bốn mô hình. Đây là thí nghiệm chuyển trọng số theo protocol GDUT, không phải lặp nguyên recipe gốc của paper; paper YOLO-DCRCF mô tả train không dùng pretrained.",
        "",
        "## 2.3. Gap Analysis",
        "",
        "Khác biệt được xác nhận: SHWD hai lớp so với GDUT năm lớp; khởi tạo pretrained/Stage1, optimizer và augmentation đã điều chỉnh theo A6. Domain, kích thước đối tượng và phân bố lớp có thể ảnh hưởng, nhưng chưa được cô lập bằng thí nghiệm riêng. Chênh lệch được báo cáo nguyên trạng, không gán toàn bộ cho kiến trúc. Bảng audit phân biệt giá trị đã ghi với giá trị thiếu trong run cũ.",
        "",
    ]
    for row in paper_rows:
        lines.append(
            f"- {row['model']} [{row['split']}]: mAP50 {100*row['experiment_mAP50']:.2f}%, paper {100*row['paper_mAP50']:.2f}%, delta {row['mAP50_delta_pp']:+.2f} điểm phần trăm."
        )
    lines.extend(
        [
            "",
            "Nguồn: https://doi.org/10.3390/jimaging11090320 ; https://doi.org/10.1038/s41598-024-68446-z",
        ]
    )
    (output / "gap_analysis.vi.md").write_text(
        "\n".join(lines) + "\n", encoding="utf-8"
    )
    print(f"Comparison complete: {output}", flush=True)
    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--prepared-dir", default="")
    parser.add_argument("--device", default="0")
    args = parser.parse_args()
    compare(args.prepared_dir, args.device)


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_data.py
"""Locate stock GDUT checkpoints and the existing train-only hardaug images."""

import csv
import json
from pathlib import Path

import yaml

from gdut_data import NAMES, sha256


def dataset_yaml(prepared, output, augmented):
    prepared, output = Path(prepared), Path(output)
    train = [str(prepared / "base/images/train")]
    if augmented:
        manifest = json.loads((prepared / "augmentation_manifest.json").read_text())
        if (
            manifest.get("image_size") != 640
            or manifest.get("copies_per_train_image") != 1
            or not manifest.get("hue_saturation_transform_removed")
            or manifest.get("validation_test_augmented")
        ):
            raise ValueError("Use the GDUT color-safe 640 train-only hardaug output.")
        base = sorted((prepared / "base/images/train").glob("*.jpg"))
        hardaug = prepared / "hardaug/images/train"
        expected = {image.stem + "_hardaug.jpg" for image in base}
        if {p.name for p in hardaug.glob("*.jpg")} != expected:
            raise ValueError("Hardaug images do not match the base training images.")
        for image in hardaug.glob("*.jpg"):
            if not (
                prepared / "hardaug/labels/train" / (image.stem + ".txt")
            ).is_file():
                raise FileNotFoundError(f"Missing hardaug label: {image.stem}")
        train.append(str(hardaug))
    output.mkdir(parents=True, exist_ok=True)
    path = output / ("gdut_aug.yaml" if augmented else "gdut_base.yaml")
    path.write_text(
        yaml.safe_dump(
            dict(
                path=str(prepared),
                names=dict(enumerate(NAMES)),
                train=train,
                val=str(prepared / "base/images/val"),
                test=str(prepared / "base/images/test"),
            )
        ),
        encoding="utf-8",
    )
    return str(path)


def find_stage1(input_root, family, dataset_hash, explicit="", allow_smoke=False):
    found = []
    for path in Path(explicit or input_root).rglob("run_manifest.json"):
        value = json.loads(path.read_text())
        if (value.get("dataset"), value.get("stage"), value.get("model")) == (
            "gdut",
            "S1",
            family,
        ):
            found.append((path, value))
    if len(found) != 1:
        raise ValueError(
            f"Mount one stock GDUT Stage1 {family} output; found {len(found)}."
        )
    path, value = found[0]
    if (
        value.get("dataset_fingerprint") != dataset_hash
        or value.get("names") != NAMES
        or value.get("imgsz") != 640
    ):
        raise ValueError("Stage1 data version, classes or image size differ.")
    if value.get("smoke") and not allow_smoke:
        raise ValueError("Use a full Stage1 checkpoint.")
    expected = dict(
        optimizer="AdamW",
        lr0=0.0003,
        momentum=0.9,
        warmup_bias_lr=0.0,
        epochs=100,
        batch=16,
        imgsz=640,
        seed=3407,
    )
    if not allow_smoke:
        for key, configured in expected.items():
            if value.get("training_args", {}).get(key) != configured:
                raise ValueError(f"Stage1 does not use the CLEAN source recipe: {key}.")
    checkpoint = (path.parent / value["checkpoint"]).resolve()
    if not checkpoint.is_relative_to(path.parent.resolve()):
        raise ValueError("Checkpoint must remain inside its run directory.")
    if not checkpoint.is_file() or sha256(checkpoint) != value["checkpoint_sha256"]:
        raise ValueError("Missing or modified Stage1 checkpoint.")
    results = path.parent / "results.csv"
    if results.is_file():
        with results.open() as stream:
            value["observed_epochs_completed"] = sum(1 for _ in csv.DictReader(stream))
    return checkpoint, value


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_environment.py
"""Runtime metadata and reproducible configuration fingerprints."""

import hashlib
import importlib
import json
import os
import platform
import torch

PIN = "8.4.115"


def fingerprint(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()


def runtime_snapshot(device, expected_gpu="Tesla T4", smoke=False):
    if device == "cpu":
        if not smoke:
            raise ValueError("Use a CUDA GPU for the full study.")
        gpus = []
    else:
        if torch.version.cuda is None:
            raise RuntimeError(
                "This session has CPU-only PyTorch. In Kaggle select Settings > "
                "Accelerator > GPU T4 x2, restart the session, then Run All."
            )
        if not torch.cuda.is_available():
            raise RuntimeError(
                "CUDA is not available in this session. Enable the Kaggle GPU "
                "accelerator and restart the session before training."
            )
        indices = [int(value) for value in str(device).split(",")]
        if len(indices) != len(set(indices)):
            raise ValueError("GPU indices must be unique.")
        count = torch.cuda.device_count()
        if any(index < 0 or index >= count for index in indices):
            raise RuntimeError(
                f"DEVICE={device!r} requests unavailable indices; this session has only "
                f"{count} GPU(s). Select GPU T4 x2 or use DEVICE='0' consistently."
            )
        gpus = []
        for index in indices:
            prop = torch.cuda.get_device_properties(index)
            if expected_gpu and expected_gpu not in prop.name:
                raise ValueError(f"Expected {expected_gpu}; found {prop.name}.")
            gpus.append(
                dict(
                    name=prop.name,
                    memory_bytes=prop.total_memory,
                    capability=[prop.major, prop.minor],
                )
            )
    versions = {}
    for name in (
        "torch",
        "torchvision",
        "ultralytics",
        "albumentations",
        "numpy",
        "scipy",
        "cv2",
        "PIL",
    ):
        try:
            versions[name] = importlib.import_module(name).__version__
        except (ImportError, AttributeError):
            if not smoke:
                raise RuntimeError(f"Install the pinned requirements; missing {name}.")
            versions[name] = "not_installed_software_check"
    if not smoke and (
        versions["ultralytics"] != PIN or versions["albumentations"] != "2.0.8"
    ):
        raise RuntimeError("Install the pinned study requirements.")
    software = dict(
        packages=versions,
        python=platform.python_version(),
        cuda=torch.version.cuda,
        cudnn=torch.backends.cudnn.version(),
        system=platform.system(),
        architecture=platform.machine(),
    )
    return dict(
        hardware=dict(gpus=gpus, cpu_architecture=platform.machine()),
        software=software,
        host=dict(cpu=platform.processor(), cpu_count=os.cpu_count()),
    )


def contract_differences(left, right, prefix=""):
    differences = []
    for key in sorted(left.keys() | right.keys()):
        path = f"{prefix}.{key}" if prefix else key
        a, b = left.get(key), right.get(key)
        if isinstance(a, dict) and isinstance(b, dict):
            differences.extend(contract_differences(a, b, path))
        elif a != b:
            differences.append(dict(field=path, left=a, right=b))
    return differences


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_launch.py
"""Stream subprocess output and preserve the complete training log."""

from collections import deque
from pathlib import Path
import subprocess

from plain_logging import plain_text


def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    arguments = list(command)
    if "-u" not in arguments[1:]:
        arguments.insert(1, "-u")
    tail = deque(maxlen=35)
    with log_path.open("w", encoding="utf-8") as log:
        with subprocess.Popen(
            arguments,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        ) as process:
            try:
                for line in process.stdout:
                    line = plain_text(line)
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                    tail.append(line)
                returncode = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
    if returncode:
        raise RuntimeError(
            f"Process stopped with exit code {returncode}. Full log: {log_path}\n"
            + "".join(tail)
        )
    return log_path


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_metrics.py
"""Export AP, precision, recall and class-wise F1 with explicit confidence rules."""

import copy
import time

import numpy as np
import torch

from custom_ablation_modules import RepConv
from gdut_data import NAMES


def f1_macro(precision, recall):
    precision, recall = np.asarray(precision), np.asarray(recall)
    values = np.divide(
        2 * precision * recall,
        precision + recall,
        out=np.zeros_like(precision, dtype=float),
        where=precision + recall > 0,
    )
    return float(values.mean()) if values.size else 0.0


def export_metrics(result, confidence=0.25):
    box = result.box
    summary = dict(
        precision=float(box.mp),
        recall=float(box.mr),
        mAP50=float(box.map50),
        mAP50_95=float(box.map),
        F1_native_macro=float(np.asarray(box.f1).mean()) if len(box.f1) else 0.0,
    )
    per_class, p_fixed, r_fixed = [], [], []
    for index, class_id in enumerate(box.ap_class_index):
        p = float(np.interp(confidence, box.px, box.p_curve[index]))
        r = float(np.interp(confidence, box.px, box.r_curve[index]))
        p_fixed.append(p)
        r_fixed.append(r)
        per_class.append(
            dict(
                class_id=int(class_id),
                name=NAMES[class_id],
                precision=float(box.p[index]),
                recall=float(box.r[index]),
                F1_native=float(box.f1[index]),
                AP50=float(box.all_ap[index, 0]),
                AP50_95=float(box.all_ap[index].mean()),
                precision_conf025=p,
                recall_conf025=r,
                F1_conf025=f1_macro([p], [r]),
            )
        )
    present = {row["class_id"] for row in per_class}
    for class_id, name in enumerate(NAMES):
        if class_id not in present:
            per_class.append(
                dict(
                    class_id=class_id,
                    name=name,
                    precision=0.0,
                    recall=0.0,
                    F1_native=0.0,
                    AP50=0.0,
                    AP50_95=0.0,
                    precision_conf025=0.0,
                    recall_conf025=0.0,
                    F1_conf025=0.0,
                )
            )
            p_fixed.append(0.0)
            r_fixed.append(0.0)
    per_class.sort(key=lambda row: row["class_id"])
    summary.update(
        precision_conf025_macro=float(np.mean(p_fixed)),
        recall_conf025_macro=float(np.mean(r_fixed)),
        F1_conf025_macro=f1_macro(p_fixed, r_fixed),
    )
    definitions = dict(
        precision_recall="Ultralytics class means at the native operating point",
        F1_native_macro="Mean class F1 at the native mean-F1 operating point; IoU 0.5",
        F1_conf025_macro="Mean class F1 interpolated from PR/confidence curves at confidence 0.25; IoU 0.5",
        AP="Ultralytics 101-point AP integration; AP50 and mean AP at IoU 0.50:0.05:0.95",
    )
    return summary, per_class, definitions


def benchmark_forward(model, device, smoke=False):
    target = torch.device("cpu" if device == "cpu" else "cuda:" + str(device))
    measured = copy.deepcopy(model).to(target).float().eval()
    converted = 0
    for layer in list(measured.modules()):
        if isinstance(layer, RepConv):
            layer.switch_to_deploy()
            converted += 1
    measured.fuse(verbose=False)
    tensor = torch.zeros(1, 3, 640, 640, device=target)
    iterations, warmup = (2, 1) if smoke else (30, 10)

    def sync():
        if target.type == "cuda":
            torch.cuda.synchronize(target)

    times = []
    with torch.inference_mode():
        for _ in range(warmup):
            measured(tensor)
        sync()
        for _ in range(iterations):
            sync()
            start = time.perf_counter()
            measured(tensor)
            sync()
            times.append(1000 * (time.perf_counter() - start))
    return dict(
        forward_ms_median=float(np.median(times)),
        forward_ms_mean=float(np.mean(times)),
        batch=1,
        imgsz=640,
        precision="float32",
        includes_preprocessing=False,
        includes_nms=False,
        warmup=warmup,
        iterations=iterations,
        device=str(target),
        hardware=torch.cuda.get_device_name(target) if target.type == "cuda" else "CPU",
        repconv_deploy_blocks=converted,
        deployment_parameters=sum(p.numel() for p in measured.parameters()),
    )


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_model.py
"""Load stock Stage1 weights before inserting the baseline modules."""

import json
import os

import torch
from ultralytics.nn.tasks import DetectionModel
from ultralytics.utils import RANK

from baseline_model import PaperBaselineTrainer, apply_baseline, module_inventory
from warm_protocol import MODELS, REVISION


def validate_stock_model(model, identifier):
    if hasattr(model, "research_variant") or hasattr(model, "baseline_id"):
        raise ValueError(
            "Use a stock Stage1 checkpoint, not an A6 or baseline checkpoint."
        )
    expected = MODELS[identifier]
    classes = {type(layer).__name__ for layer in model.model}
    family = (
        "yolo11" if "C2PSA" in classes else "yolov8" if "C2f" in classes else "unknown"
    )
    expected_family = "yolo11" if expected.startswith("yolo11") else "yolov8"
    stem = model.model[0].conv.out_channels
    if family != expected_family or stem != (16 if expected == "yolo11n" else 32):
        raise ValueError(f"Expected the {expected} stock Stage1 scale.")
    if model.model[-1].nc != 5:
        raise ValueError("Stage1 must use the five GDUT classes.")


def transfer_baseline(model, identifier):
    validate_stock_model(model, identifier)
    previous_parameters = {
        id(value): (name, value) for name, value in model.named_parameters()
    }
    previous_buffers = {
        id(value): (name, value) for name, value in model.named_buffers()
    }
    pooling = model.model[9] if identifier.startswith("yolov8") else None
    apply_baseline(model, identifier)
    explicit = {}
    if pooling is not None:
        target = model.model[9].branches[0]
        with torch.no_grad():
            target[0].weight.copy_(pooling.cv1.conv.weight)
            target[1].load_state_dict(pooling.cv1.bn.state_dict())
        target[1].eps, target[1].momentum = pooling.cv1.bn.eps, pooling.cv1.bn.momentum
        explicit["model.9.branches.0.0.weight"] = "model.9.cv1.conv.weight"
        for key in pooling.cv1.bn.state_dict():
            explicit["model.9.branches.0.1." + key] = "model.9.cv1.bn." + key
    mapping, initialized, transferred_elements = [], [], 0
    for name, parameter in model.named_parameters():
        retained = previous_parameters.get(id(parameter))
        source_name = retained[0] if retained is not None else explicit.get(name)
        if source_name is not None:
            mapping.append(
                dict(
                    target=name,
                    source=source_name,
                    shape=list(parameter.shape),
                    kind="parameter",
                )
            )
            transferred_elements += parameter.numel()
        else:
            initialized.append(dict(name=name, shape=list(parameter.shape)))
    for name, buffer in model.named_buffers():
        retained = previous_buffers.get(id(buffer))
        source_name = retained[0] if retained is not None else explicit.get(name)
        if source_name is not None:
            mapping.append(
                dict(
                    target=name,
                    source=source_name,
                    shape=list(buffer.shape),
                    kind="buffer",
                )
            )
    audit = dict(
        source_model=MODELS[identifier],
        transferred=mapping,
        newly_initialized_parameters=initialized,
        transferred_parameter_elements=transferred_elements,
        total_parameter_elements=sum(p.numel() for p in model.parameters()),
        new_modules_initialization="native baseline initializers",
    )
    model.transfer_audit = audit
    model.warm_revision = REVISION
    return model, audit


class WarmTrainer(PaperBaselineTrainer):
    baseline_id = "yolo11_dcrcf"

    def get_model(self, cfg=None, weights=None, verbose=True):
        if weights is None:
            raise ValueError("A stock GDUT Stage1 checkpoint is required.")
        validate_stock_model(weights, self.baseline_id)
        model = self.set_model_names_for_load(
            DetectionModel(
                cfg, nc=self.data["nc"], ch=self.data["channels"], verbose=verbose
            )
        )
        model.load(weights)
        model, audit = transfer_baseline(model, self.baseline_id)
        source_state, target_state = weights.state_dict(), model.state_dict()
        for item in audit["transferred"]:
            target = target_state[item["target"]]
            expected = source_state[item["source"]].to(
                device=target.device, dtype=target.dtype
            )
            if not torch.equal(target, expected):
                raise ValueError(f"Transferred tensor differs: {item['target']}")
        report = dict(
            baseline_id=self.baseline_id,
            revision=REVISION,
            rank=os.environ.get("RANK", "-1"),
            modules=module_inventory(model),
            transfer=audit,
        )
        self.save_dir.mkdir(parents=True, exist_ok=True)
        (self.save_dir / f"architecture_rank_{report['rank']}.json").write_text(
            json.dumps(report, indent=2), encoding="utf-8"
        )
        return model

    def final_eval(self):
        super().final_eval()
        if RANK in {-1, 0}:
            path = self.save_dir / "training_complete.json"
            value = json.loads(path.read_text())
            value["revision"] = REVISION
            path.write_text(json.dumps(value, indent=2), encoding="utf-8")


class WarmDCRCFTrainer(WarmTrainer):
    baseline_id = "yolo11_dcrcf"


class WarmDWRTrainer(WarmTrainer):
    baseline_id = "yolov8_dwr_aspp_nwd"


TRAINERS = {"yolo11_dcrcf": WarmDCRCFTrainer, "yolov8_dwr_aspp_nwd": WarmDWRTrainer}


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_protocol.py
"""Stage1 and fine-tuning settings shared with the CLEAN A6 recipe."""

REVISION = "gdut_stage1_baselines_v1"
MODELS = {"yolo11_dcrcf": "yolo11n", "yolov8_dwr_aspp_nwd": "yolov8s"}
COMMON = {
    "epochs": 100,
    "batch": 16,
    "imgsz": 640,
    "seed": 3407,
    "optimizer": "AdamW",
    "lr0": 0.0001,
    "lrf": 0.01,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "pretrained": True,
    "resume": False,
    "patience": 50,
    "deterministic": True,
    "workers": 4,
    "amp": True,
    "compile": False,
    "cos_lr": False,
    "warmup_epochs": 3.0,
    "warmup_momentum": 0.8,
    "warmup_bias_lr": 0.0,
    "nbs": 64,
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5,
    "hsv_h": 0.0,
    "hsv_s": 0.0,
    "hsv_v": 0.4,
    "bgr": 0.0,
    "degrees": 0.0,
    "translate": 0.1,
    "scale": 0.5,
    "shear": 0.0,
    "perspective": 0.0,
    "flipud": 0.0,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "close_mosaic": 10,
    "mixup": 0.0,
    "cutmix": 0.0,
    "copy_paste": 0.0,
    "save": True,
    "save_period": -1,
    "plots": True,
    "cache": False,
    "val": True,
    "rect": False,
    "fraction": 1.0,
    "multi_scale": False,
}
EVALUATION = dict(
    imgsz=640,
    batch=16,
    conf=0.001,
    iou=0.7,
    max_det=300,
    workers=0,
    half=False,
    quantize=None,
    augment=False,
    plots=True,
)


def training_settings(device="0,1", stage1=False, smoke=False):
    values = dict(COMMON, device=device)
    if stage1:
        values["lr0"] = 0.0003
    if smoke:
        values.update(epochs=1, batch=2, imgsz=64, workers=0, plots=False, amp=False)
    return values


In [ ]:
%%writefile /kaggle/working/stage1_baseline_code/warm_runner.py
"""Train stock nano Stage1 and fine-tune the baseline graphs from GDUT weights."""

import argparse
import gc
import json
from pathlib import Path
import random

import numpy as np
import torch
import ultralytics
from ultralytics import YOLO
import yaml

from baseline_model import assert_inventory, module_inventory
from gdut_data import prepared_dataset, sha256
from plain_logging import configure_plain_logging
from warm_data import dataset_yaml, find_stage1
from warm_environment import PIN, runtime_snapshot
from warm_metrics import export_metrics
from warm_model import TRAINERS
from warm_protocol import COMMON, EVALUATION, MODELS, REVISION, training_settings

configure_plain_logging()
CODE_DIR = Path(__file__).resolve().parent


def write_json(path, value):
    Path(path).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def set_seed():
    random.seed(3407)
    np.random.seed(3407)
    torch.manual_seed(3407)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(3407)


def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def check_version():
    if ultralytics.__version__ != PIN:
        raise RuntimeError(f"Use ultralytics=={PIN}.")


def validate_saved_args(path, configured):
    actual = yaml.safe_load(Path(path).read_text())
    for key in COMMON:
        if actual.get(key) != configured[key]:
            raise ValueError(f"Training setting changed: {key}.")
    return actual


def validation(model, data, root, device, smoke):
    settings = dict(EVALUATION)
    if smoke:
        settings.update(imgsz=64, batch=2, plots=False)
    result = model.val(
        data=data,
        split="val",
        device=device.split(",")[0],
        project=str(root),
        name="val_check",
        **settings,
    )
    values, classes, definitions = export_metrics(result)
    write_json(
        root / "validation_details.json",
        dict(metrics=values, per_class=classes, definitions=definitions),
    )
    return values


def train_stage1(
    prepared_dir="",
    device="0,1",
    working="/kaggle/working",
    expected_gpu="Tesla T4",
    smoke=False,
    source_model="yolo11n.pt",
):
    check_version()
    prepared, metadata, dataset_hash = prepared_dataset(prepared_dir)
    runtime = runtime_snapshot(device, expected_gpu, smoke)
    root = Path(working) / "stage1_runs/gdut_S1_yolo11n"
    if root.exists():
        raise FileExistsError(f"Use a fresh Stage1 directory: {root}")
    data = dataset_yaml(prepared, Path(working) / "stage1_config", False)
    args = training_settings(device, stage1=True, smoke=smoke)
    args.update(data=data, project=str(root.parent), name=root.name, exist_ok=False)
    set_seed()
    stock = YOLO(source_model)
    stock.train(**args)
    del stock
    release()
    best = root / "weights/best.pt"
    model = YOLO(str(best))
    actual_args = validate_saved_args(root / "args.yaml", args)
    metrics = validation(model, data, root, device, smoke)
    write_json(
        root / "run_manifest.json",
        dict(
            dataset="gdut",
            stage="S1",
            model="yolo11n",
            names=metadata["names"],
            imgsz=640,
            smoke=smoke,
            checkpoint="weights/best.pt",
            checkpoint_sha256=sha256(best),
            dataset_fingerprint=dataset_hash,
            source_pretrained_model="yolo11n.pt",
            training_args=actual_args,
            validation={
                "metrics/precision(B)": metrics["precision"],
                "metrics/recall(B)": metrics["recall"],
                "metrics/mAP50(B)": metrics["mAP50"],
                "metrics/mAP50-95(B)": metrics["mAP50_95"],
            },
            runtime=runtime,
            recipe="conservative_v2_nano_scale",
            initialization="stock_pretrained",
            warm_revision=REVISION,
        ),
    )
    print(f"Stock YOLO11n Stage1 complete: {root}", flush=True)
    return root


def train_baseline(
    identifier,
    prepared_dir="",
    checkpoint_dir="",
    device="0,1",
    working="/kaggle/working",
    expected_gpu="Tesla T4",
    smoke=False,
    input_root="/kaggle/input",
):
    check_version()
    prepared, metadata, dataset_hash = prepared_dataset(prepared_dir)
    weight, source = find_stage1(
        input_root, MODELS[identifier], dataset_hash, checkpoint_dir, smoke
    )
    runtime = runtime_snapshot(device, expected_gpu, smoke)
    root = Path(working) / "transfer_runs" / ("gdut_" + identifier)
    if root.exists():
        raise FileExistsError(f"Use a fresh fine-tuning directory: {root}")
    data = dataset_yaml(prepared, Path(working) / "transfer_config", True)
    args = training_settings(device, smoke=smoke)
    args.update(
        model=str(weight),
        data=data,
        project=str(root.parent),
        name=root.name,
        exist_ok=False,
    )
    set_seed()
    trainer = TRAINERS[identifier](overrides=args)
    trainer.train()
    del trainer
    release()
    marker = json.loads((root / "training_complete.json").read_text())
    if not 1 <= marker["epochs_completed"] <= args["epochs"]:
        raise ValueError("Invalid completed epoch count.")
    actual_args = validate_saved_args(root / "args.yaml", args)
    best = root / "weights/best.pt"
    model = YOLO(str(best))
    if model.model.warm_revision != REVISION or model.model.baseline_id != identifier:
        raise ValueError("Checkpoint and fine-tuning run disagree.")
    assert_inventory(model.model)
    modules, audit = module_inventory(model.model), model.model.transfer_audit
    parameters = sum(p.numel() for p in model.model.parameters())
    ranks = (
        [-1] if device == "cpu" or "," not in device else range(len(device.split(",")))
    )
    for rank in ranks:
        record = json.loads((root / f"architecture_rank_{rank}.json").read_text())
        if record["modules"] != modules:
            raise ValueError(
                "Distributed workers constructed different baseline graphs."
            )
    metrics = validation(model, data, root, device, smoke)
    write_json(root / "transfer_audit.json", audit)
    write_json(
        root / "run_manifest.json",
        dict(
            dataset="gdut",
            names=metadata["names"],
            baseline_id=identifier,
            warm_revision=REVISION,
            model=MODELS[identifier],
            imgsz=640,
            stage="baseline_finetune",
            smoke=smoke,
            checkpoint="weights/best.pt",
            checkpoint_sha256=sha256(best),
            dataset_fingerprint=dataset_hash,
            augmentation_manifest_sha256=sha256(
                prepared / "augmentation_manifest.json"
            ),
            source_checkpoint_sha256=source["checkpoint_sha256"],
            source_manifest=source,
            initialization="stock_GDUT_Stage1",
            hardaug=True,
            training_args=actual_args,
            epochs_completed=marker["epochs_completed"],
            best_epoch=marker.get("best_epoch"),
            validation=metrics,
            modules=modules,
            transfer=audit,
            parameters=parameters,
            runtime=runtime,
            code_sha256={p.name: sha256(p) for p in CODE_DIR.glob("*.py")},
        ),
    )
    del model
    release()
    print(f"Baseline fine-tuning complete: {root}", flush=True)
    return root


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--stage1-nano", action="store_true")
    parser.add_argument("--baseline", choices=MODELS)
    parser.add_argument("--prepared-dir", default="")
    parser.add_argument("--checkpoint-dir", default="")
    parser.add_argument("--device", default="0,1")
    parser.add_argument("--expected-gpu", default="Tesla T4")
    parser.add_argument("--smoke", action="store_true")
    args = parser.parse_args()
    if args.stage1_nano:
        train_stage1(
            args.prepared_dir,
            args.device,
            expected_gpu=args.expected_gpu,
            smoke=args.smoke,
        )
    elif args.baseline:
        train_baseline(
            args.baseline,
            args.prepared_dir,
            args.checkpoint_dir,
            args.device,
            expected_gpu=args.expected_gpu,
            smoke=args.smoke,
        )
    else:
        parser.error("Select --stage1-nano or --baseline.")


In [ ]:
import json

paper_references = {
    "yolo11_dcrcf": {
        "title": "YOLO-DCRCF",
        "doi": "10.3390/jimaging11090320",
        "url": "https://www.mdpi.com/2313-433X/11/9/320",
        "table": "Table 2; training: Section 5.1; dataset: Section 3",
        "dataset": "SHWD",
        "classes": ["helmet", "head without helmet"],
        "split": "5306 train / 1593 validation / 682 test images; cross-validation also reported",
        "metrics": {
            "precision": 0.909,
            "recall": 0.885,
            "mAP50": 0.927,
            "mAP50_95": 0.606,
        },
        "training": {
            "epochs": 300,
            "batch": 32,
            "imgsz": 640,
            "optimizer": "SGD",
            "pretrained": False,
            "lr0": None,
            "seed": None,
        },
        "hardware": "GeForce RTX 4090",
        "software": "Python 3.10; PyTorch 2.0.1; Windows 11",
        "preprocessing": "SHWD split and source-image annotations",
        "evidence": "Published paper; locally archived PDF and source audit",
    },
    "yolov8_dwr_aspp_nwd": {
        "title": "An improved YOLOv8 safety helmet wearing detection network",
        "doi": "10.1038/s41598-024-68446-z",
        "url": "https://www.nature.com/articles/s41598-024-68446-z",
        "table": "Table 7; training: Table 2; dataset: Datasets section",
        "dataset": "SHWD with augmentation",
        "classes": ["helmet", "head without helmet"],
        "split": "10581 augmented images; 80/20 train/val split described in paper",
        "metrics": {
            "precision": 0.918,
            "recall": 0.866,
            "mAP50": 0.92,
            "mAP50_95": 0.622,
        },
        "training": {
            "epochs": 100,
            "batch": 16,
            "imgsz": None,
            "optimizer": None,
            "pretrained": None,
            "lr0": 0.01,
            "momentum": 0.937,
            "weight_decay": 0.0005,
            "seed": None,
        },
        "hardware": "GeForce RTX 3090",
        "software": "Python 3.8.15; PyTorch 1.13.1; CUDA 11.6; Windows 10",
        "preprocessing": "Offline expansion of SHWD from 7581 to 10581 images",
        "evidence": "Published paper; locally archived PDF and source audit",
    },
}
(CODE_DIR / "paper_references.json").write_text(
    json.dumps(paper_references, indent=2), encoding="utf-8"
)


In [ ]:
PREPARED_DIR = ""
DEVICE = "0,1"
EXPECTED_GPU = "Tesla T4"
SMOKE = False

command = [
    sys.executable,
    str(CODE_DIR / "warm_runner.py"),
    "--stage1-nano",
    "--prepared-dir",
    PREPARED_DIR,
    "--device",
    DEVICE,
    "--expected-gpu",
    EXPECTED_GPU,
]
if SMOKE:
    command.append("--smoke")
from warm_environment import runtime_snapshot
from warm_launch import run_logged

runtime_snapshot(DEVICE, EXPECTED_GPU, SMOKE)
run_logged(command, Path("/kaggle/working/stage1_yolo11n.log"))
